In [2]:
# Training Arabic Sentiment Model Notebook
import pandas as pd
import joblib
import stanza
import nltk
from Preprocessing_pipeline import normalize_arabic
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from nltk.corpus import stopwords

# 1. Download models and resources once at class load time
stanza.download('ar')
nltk.download('stopwords', quiet=True)
    
# Initialize the Stanza pipeline and stopwords set once in memory
nlp = stanza.Pipeline('ar', processors='tokenize,mwt,pos,lemma', verbose=False)
arabic_stopwords = set(stopwords.words('arabic'))


df = pd.read_csv('../data/arabic_reviews.csv')
df = df.head(1000).copy()
df = df.dropna(subset=['review_description', 'rating']).copy()
df['clean_text'] = df['review_description'].apply(
    normalize_arabic,
    nlp=nlp,
    arabic_stopwords=arabic_stopwords
    )


def map_sentiment(val):
    val = str(val).strip().lower()
    if val in ['positive', '5', '4', 'ممتاز', 'إيجابي']:
        return 'Positive'
    elif val in ['negative', '1', '2', 'سيء', 'سلبي']:
        return 'Negative'
    return 'Neutral'

df['sentiment'] = df['rating'].apply(map_sentiment)

X_train, X_test, y_train, y_test = train_test_split(df['clean_text'], df['sentiment'], test_size=0.2, random_state=42)

model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=15000)),
    ('clf', LogisticRegression(C=2.0, max_iter=1000))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

joblib.dump(model, 'Arabic_model_weights.pkl')
print("Model saved to Arabic_model_weights.pkl")


              precision    recall  f1-score   support

    Negative       0.80      0.99      0.89       139
     Neutral       0.00      0.00      0.00         5
    Positive       0.93      0.46      0.62        56

    accuracy                           0.82       200
   macro avg       0.58      0.49      0.50       200
weighted avg       0.82      0.82      0.79       200

Model saved to Arabic_model_weights.pkl


d:\AI\OmarNlp\NLP-Course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\AI\OmarNlp\NLP-Course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\AI\OmarNlp\NLP-Course\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
